# cluster01 bike_change 모델링

## 용어 설명

- `cluster01`(아침 도착 업무 집중형): 대표 15개 중 가장 어려운 군집으로 확인된 그룹
- `bike_change`(시간별 대여량 변화량): 현재 시간 `rental_count`에서 직전 시간 `rental_count`를 뺀 값
- `baseline`(기준선 모델): 군집 안에서 먼저 비교하는 기본 모델
- `sign_accuracy`(부호 일치율): 예측값과 실제값의 증가/감소 방향이 같은 비율

목표:
- `cluster01`에서 `LightGBM_RMSE`와 `LinearRegression`을 먼저 비교한다.
- 이후 subset 실험이나 추가 보강 전에 군집 기준선을 만든다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
TRAIN_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_test_2025.csv'
OUTPUT_DIR = WORK_DIR / 'output/data'

TARGET_CLUSTER = 1
TARGET_STATION_GROUP = '아침 도착 업무 집중형'
RANDOM_STATE = 42


In [2]:
FEATURE_COLUMNS = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h', 'is_weekend', 'is_night_hour',
    'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos', 'commute_morning_flag',
    'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
    'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]
categorical_features = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']
numeric_features = [c for c in FEATURE_COLUMNS if c not in categorical_features]
target_col = 'bike_change'


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_group = train_raw.loc[train_raw['cluster'] == TARGET_CLUSTER].copy()
test_group = test_raw.loc[test_raw['cluster'] == TARGET_CLUSTER].copy()

train_2023 = train_group.loc[train_group['date'] < '2024-01-01'].copy()
valid_2024 = train_group.loc[train_group['date'] >= '2024-01-01'].copy()
test_2025 = test_group.copy()

print('train_group shape:', train_group.shape)
print('test_group shape:', test_group.shape)
print(train_group[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').to_string(index=False))


train_group shape: (52128, 40)
test_group shape: (8592, 40)
 station_id   station_name
       2323  주식회사 오뚜기 정문 앞
       2348   포스코사거리(기업은행)
       2377       수서역 5번출구


In [4]:
def evaluate_regression(y_true, y_pred):
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'sign_accuracy': float(sign_accuracy),
    }

def build_linear_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])

def build_lgbm_model():
    return LGBMRegressor(
        objective='regression',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


In [5]:
results = []

X_train = train_2023[FEATURE_COLUMNS]
y_train = train_2023[target_col]
X_valid = valid_2024[FEATURE_COLUMNS]
y_valid = valid_2024[target_col]

linear_model = build_linear_pipeline()
linear_model.fit(X_train, y_train)
linear_metrics = evaluate_regression(y_train, linear_model.predict(X_train))
results.append({'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': 'LinearRegression', 'split': 'train_2023', **linear_metrics})
linear_metrics = evaluate_regression(y_valid, linear_model.predict(X_valid))
results.append({'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': 'LinearRegression', 'split': 'validation_2024', **linear_metrics})

lgbm_model = build_lgbm_model()
lgbm_model.fit(X_train, y_train, categorical_feature=categorical_features)
lgbm_metrics = evaluate_regression(y_train, lgbm_model.predict(X_train))
results.append({'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': 'LightGBM_RMSE', 'split': 'train_2023', **lgbm_metrics})
lgbm_metrics = evaluate_regression(y_valid, lgbm_model.predict(X_valid))
results.append({'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': 'LightGBM_RMSE', 'split': 'validation_2024', **lgbm_metrics})

validation_results = pd.DataFrame(results)
validation_results


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001812 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1860
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34
[LightGBM] [Info] Start training from score 0.000078


,cluster,station_group,model,split,rmse,mae,r2,sign_accuracy
0,1,아침 도착 업무 집중형,LinearRegression,train_2023,1.695834,1.060430,0.479234,0.712019
1,1,아침 도착 업무 집중형,LinearRegression,validation_2024,1.704961,1.041570,0.493119,0.704577
2,1,아침 도착 업무 집중형,LightGBM_RMSE,train_2023,1.077477,0.721581,0.789771,0.867241
3,1,아침 도착 업무 집중형,LightGBM_RMSE,validation_2024,1.559604,0.920490,0.575863,0.846008


In [6]:
best_model_name = validation_results[validation_results['split'] == 'validation_2024'].sort_values('rmse').iloc[0]['model']
print('best_model_name:', best_model_name)

refit_train = train_group.copy()
X_refit = refit_train[FEATURE_COLUMNS]
y_refit = refit_train[target_col]
X_test = test_2025[FEATURE_COLUMNS]
y_test = test_2025[target_col]

if best_model_name == 'LinearRegression':
    best_model = build_linear_pipeline()
    best_model.fit(X_refit, y_refit)
else:
    best_model = build_lgbm_model()
    best_model.fit(X_refit, y_refit, categorical_feature=categorical_features)

train_metrics = evaluate_regression(y_refit, best_model.predict(X_refit))
train_results = pd.DataFrame([{'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': best_model_name, 'split': 'train_full_refit', **train_metrics}])

test_metrics = evaluate_regression(y_test, best_model.predict(X_test))
test_results = pd.DataFrame([{'cluster': TARGET_CLUSTER, 'station_group': TARGET_STATION_GROUP, 'model': best_model_name, 'split': 'test_2025_refit', **test_metrics}])

final_results = pd.concat([validation_results, train_results, test_results], ignore_index=True)
final_results


best_model_name: LightGBM_RMSE
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007003 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1905
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34
[LightGBM] [Info] Start training from score 0.000019


,cluster,station_group,model,split,rmse,mae,r2,sign_accuracy
0,1,아침 도착 업무 집중형,LinearRegression,train_2023,1.695834,1.060430,0.479234,0.712019
1,1,아침 도착 업무 집중형,LinearRegression,validation_2024,1.704961,1.041570,0.493119,0.704577
2,1,아침 도착 업무 집중형,LightGBM_RMSE,train_2023,1.077477,0.721581,0.789771,0.867241
3,1,아침 도착 업무 집중형,LightGBM_RMSE,validation_2024,1.559604,0.920490,0.575863,0.846008
4,1,아침 도착 업무 집중형,LightGBM_RMSE,train_full_refit,1.202564,0.783202,0.743123,0.870933
5,1,아침 도착 업무 집중형,LightGBM_RMSE,test_2025_refit,1.571622,0.953124,0.624259,0.863943


In [7]:
final_metrics = pd.concat([validation_results, train_results, test_results], ignore_index=True)
metrics_path = OUTPUT_DIR / 'ddri_bike_change_cluster01_model_metrics.csv'
final_metrics.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
final_metrics


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_cluster01_model_metrics.csv


,cluster,station_group,model,split,rmse,mae,r2,sign_accuracy
0,1,아침 도착 업무 집중형,LinearRegression,train_2023,1.695834,1.060430,0.479234,0.712019
1,1,아침 도착 업무 집중형,LinearRegression,validation_2024,1.704961,1.041570,0.493119,0.704577
2,1,아침 도착 업무 집중형,LightGBM_RMSE,train_2023,1.077477,0.721581,0.789771,0.867241
3,1,아침 도착 업무 집중형,LightGBM_RMSE,validation_2024,1.559604,0.920490,0.575863,0.846008
4,1,아침 도착 업무 집중형,LightGBM_RMSE,train_full_refit,1.202564,0.783202,0.743123,0.870933
5,1,아침 도착 업무 집중형,LightGBM_RMSE,test_2025_refit,1.571622,0.953124,0.624259,0.863943
